# Friction-coupling sweep — held motor vs. swept motor (coupled bench rig)

**Rig:** the coupled dev-bench rig — two Nezha motors on **ports 3 and 4** whose
shafts are linked by a friction coupling, so torque applied by one shows up as
load (or assist) on the other. Same rig as `tests/bench/pid_hold_speed.py` and
`tests/bench/ratio_governor_curve.py`.

**Experiment:** hold **motor 3** open-loop at a constant **+50 % duty**, sweep
**motor 4** open-loop from **−50 % to +50 % duty** in 5 % steps. At each step,
settle, then sample both motors' measured velocity from `DEV M <n> STATE`
(`vel=`, mm/s — protocol v2 §16). Because the shafts are friction-coupled,
motor 3's *measured* speed varies with motor 4's commanded duty even though
motor 3's own command never changes — opposing torque drags it down, assisting
torque pushes it up. The plot at the bottom is that transfer curve. (Whether
positive duty on port 4 assists or opposes port 3 depends on how the rig is
geared — the sweep reveals the sign; the notebook doesn't assume it.)

**Requirements:** dev-bench firmware (`ROBOT_DEV_BUILD=1`, `DEV` family) flashed,
robot connected over direct USB. HITL — a person should be present, per
`tests/CLAUDE.md`.

**Safety:** the serial-silence watchdog is widened to 3 s for the session and
restored to the boot default (1 s) afterward; the sweep cell always sends
`DEV STOP` in its `finally`, so motors are never left running on an exception
or interrupt.

**Run headless:**
```
uv run jupyter nbconvert --to notebook --execute --inplace tests/notebooks/friction_coupling_sweep.ipynb
```


In [ ]:
import pathlib
import statistics
import time

import pandas as pd

from robot_radio.io.serial_conn import SerialConnection
from robot_radio.robot.protocol import NezhaProtocol, parse_response

PORT = "/dev/cu.usbmodem2121102"  # direct-USB serial port (same as tests/bench scripts)

HOLD_PORT = 3            # motor held at constant duty
SCAN_PORT = 4            # motor swept across the duty range
HOLD_DUTY = 50.0         # [%] constant open-loop duty on HOLD_PORT
SCAN_DUTY_START = -50.0  # [%]
SCAN_DUTY_STOP = 50.0    # [%]
SCAN_DUTY_STEP = 5.0     # [%]

SPINUP_TIME = 2.0        # [s] hold-motor spin-up before the first scan point
SETTLE_TIME = 1.5        # [s] wait after each duty change before sampling
SAMPLE_TIME = 1.2        # [s] sampling window per scan point
SAMPLE_PERIOD = 0.12     # [s] between STATE polls inside the window

SESSION_WATCHDOG_WINDOW = 3000  # [ms] widened for this session
BOOT_WATCHDOG_WINDOW = 1000     # [ms] firmware default -- restored on cleanup

OUT_DIR = pathlib.Path("out")   # CSV + PNG land here (tests/notebooks/out/)


In [ ]:
def dev_send(proto, cmd, timeout=500, retries=6):  # [ms] timeout
    """Send one DEV command, retrying on a totally silent reply.

    The direct-USB CDC link occasionally, burstily drops replies outright
    (see tests/bench/dev_exercise.py's dev_send). Safe to retry
    unconditionally: every command here is a pure query (STATE) or an
    idempotent absolute-value write (DUTY/WD/STOP).
    """
    for attempt in range(retries):
        resp = proto.send(cmd, timeout)
        for raw in resp.get("responses", []):
            r = parse_response(raw)
            if r is not None and r.tag in ("OK", "ERR"):
                return r
        if attempt < retries - 1:
            time.sleep(0.1)
    return None


def kv_float(r, key):
    if r is None or key not in r.kv:
        return None
    try:
        return float(r.kv[key])
    except ValueError:
        return None


conn = SerialConnection(port=PORT)
info = conn.connect()
if "error" in info:
    raise RuntimeError(f"connect failed: {info['error']}")
print(f"connected: mode={info.get('mode')}")
proto = NezhaProtocol(conn)

preflight = dev_send(proto, f"DEV M {HOLD_PORT} STATE")
if preflight is None or preflight.tag != "OK":
    raise RuntimeError(
        "DEV family not answering -- is the dev-bench firmware "
        f"(ROBOT_DEV_BUILD) flashed? last reply: {preflight.raw if preflight else None}")
print(preflight.raw)


In [ ]:
step_count = int(round((SCAN_DUTY_STOP - SCAN_DUTY_START) / SCAN_DUTY_STEP)) + 1
scan_duties = [SCAN_DUTY_START + i * SCAN_DUTY_STEP for i in range(step_count)]  # [%]


def poll_keepalive(duration):  # [s]
    """Sleep for `duration` while polling STATE so the watchdog stays fed."""
    deadline = time.monotonic() + duration
    while time.monotonic() < deadline:
        time.sleep(min(0.25, max(0.0, deadline - time.monotonic())))
        dev_send(proto, f"DEV M {HOLD_PORT} STATE")


rows = []
try:
    dev_send(proto, f"DEV WD {SESSION_WATCHDOG_WINDOW}")
    dev_send(proto, "DEV STOP")  # known-clean start: all motors neutral

    r = dev_send(proto, f"DEV M {HOLD_PORT} DUTY {HOLD_DUTY:g}")
    print(f"hold: {r.raw if r else '(no reply)'}")
    poll_keepalive(SPINUP_TIME)

    for duty in scan_duties:
        # Re-assert the hold each step (idempotent) so a missed watchdog
        # neutral event can't silently zero the held motor mid-sweep.
        dev_send(proto, f"DEV M {HOLD_PORT} DUTY {HOLD_DUTY:g}")
        dev_send(proto, f"DEV M {SCAN_PORT} DUTY {duty:g}")
        poll_keepalive(SETTLE_TIME)

        vel_hold, vel_scan = [], []  # [mm/s]
        applied_hold = None          # duty fraction echoed by firmware
        wedged_any = False
        deadline = time.monotonic() + SAMPLE_TIME
        while time.monotonic() < deadline:
            h = dev_send(proto, f"DEV M {HOLD_PORT} STATE")
            s = dev_send(proto, f"DEV M {SCAN_PORT} STATE")
            vh, vs = kv_float(h, "vel"), kv_float(s, "vel")
            if vh is not None:
                vel_hold.append(vh)
            if vs is not None:
                vel_scan.append(vs)
            a = kv_float(h, "applied")
            if a is not None:
                applied_hold = a
            wedged_any = (wedged_any
                          or kv_float(h, "wedged") == 1.0
                          or kv_float(s, "wedged") == 1.0)
            time.sleep(SAMPLE_PERIOD)

        row = {
            "duty_scan": duty,                                                          # [%]
            "vel_hold_mean": statistics.fmean(vel_hold) if vel_hold else float("nan"),  # [mm/s]
            "vel_hold_std": statistics.stdev(vel_hold) if len(vel_hold) > 1 else 0.0,   # [mm/s]
            "vel_scan_mean": statistics.fmean(vel_scan) if vel_scan else float("nan"),  # [mm/s]
            "vel_scan_std": statistics.stdev(vel_scan) if len(vel_scan) > 1 else 0.0,   # [mm/s]
            "applied_hold": applied_hold,
            "wedged_any": wedged_any,
            "n_hold": len(vel_hold),
            "n_scan": len(vel_scan),
        }
        rows.append(row)
        print(f"  duty{SCAN_PORT}={duty:+6.1f}%  "
              f"vel{HOLD_PORT}={row['vel_hold_mean']:8.1f} mm/s (n={row['n_hold']})  "
              f"vel{SCAN_PORT}={row['vel_scan_mean']:8.1f} mm/s (n={row['n_scan']})"
              f"{'  WEDGED' if wedged_any else ''}")
finally:
    dev_send(proto, "DEV STOP")
    dev_send(proto, f"DEV WD {BOOT_WATCHDOG_WINDOW}")
    print("cleanup: motors stopped, watchdog restored")


In [ ]:
df = pd.DataFrame(rows)
OUT_DIR.mkdir(exist_ok=True)
csv_path = OUT_DIR / "friction_coupling_sweep.csv"
df.to_csv(csv_path, index=False)
print(f"saved {csv_path} ({len(df)} points)")
df


In [ ]:
import matplotlib.pyplot as plt

BLUE = "#2a78d6"  # held motor
AQUA = "#1baf7a"  # swept motor

fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="white")
ax.set_facecolor("#fcfcfb")

ax.axhline(0, color="#d8d7d3", lw=1, zorder=1)
ax.axvline(0, color="#d8d7d3", lw=1, zorder=1)

ax.errorbar(df["duty_scan"], df["vel_hold_mean"], yerr=df["vel_hold_std"],
            color=BLUE, marker="o", ms=6, lw=2, capsize=3, zorder=3,
            label=f"motor {HOLD_PORT} (held at {HOLD_DUTY:g}% duty)")
ax.errorbar(df["duty_scan"], df["vel_scan_mean"], yerr=df["vel_scan_std"],
            color=AQUA, marker="s", ms=5, lw=1.5, capsize=3, zorder=2,
            label=f"motor {SCAN_PORT} (swept)")

wedged_pts = df[df["wedged_any"]]
if not wedged_pts.empty:
    ax.scatter(wedged_pts["duty_scan"], wedged_pts["vel_hold_mean"],
               marker="x", color="#d03b3b", s=70, zorder=4,
               label="wedged flag seen")

ax.set_xlabel(f"Motor {SCAN_PORT} commanded duty [%]")
ax.set_ylabel("Measured velocity [mm/s]")
ax.set_title(f"Friction coupling: motor {HOLD_PORT} speed vs. motor {SCAN_PORT} duty\n"
             f"(motor {HOLD_PORT} held open-loop at {HOLD_DUTY:g}% duty, coupled rig)")
ax.grid(True, color="#e8e7e3", lw=0.8, zorder=0)
ax.legend(frameon=False)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_DIR / "friction_coupling_sweep.png", dpi=150)
plt.show()


In [ ]:
if conn.is_open:
    conn.disconnect()
    print("serial closed")
